In [1]:
# THIS CODE USES NUMPY; FOR FEWER THAN 500 GRIDPOINTS
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from PIL import Image
from numba import jit, prange

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
from scipy.signal import find_peaks
from scipy.integrate import quad
from scipy.optimize import root_scalar
from matplotlib.animation import PillowWriter, FuncAnimation
import ipywidgets as widgets
from IPython.display import display

print("Fin")

Fin


In [2]:
# Independent parameters (free to edit)

Na = 0.5 # Units: M 
T = 303.15 # Units: K
valence = 4
duration = 250 * 10**5 # In timesteps of dt
gridpoints = 1024 # Number of points
dx = 10 # Units: nm
dt = 1.E-5 # Units: sec
rho_mean = 9E-5 # Initial mean density of nanostar A, found by spinodal (rho dense + rho dilute)/2 for the value of T used
save_interval = 10**5

grid_length = dx * gridpoints # Total length (nm)
inv_dx2 = 1.0 / (dx * dx)

# Establishes constants
K = 1.0E6 # Units: nm^5 
M = 1 # Units: (nm s)^-1
B2 = 2190 # Units: nm^3
vb = 1.66 # Units: nm^3
kB = 1.314E-23 * 0.24 # Units: cal/K (1J=0.24cal)
mol = 6.02E23
dHa = -42000 # Units: cal/mol 
dS1 = 1.84 * np.log(Na) # Units: cal/mol K
dS0 = -120 # Units: cal/mol K at 1M NaCl
floor = 1E-12 # Minimum value for arrays
num_saves = duration // save_interval + 1 # Number of saved values

Da = vb * np.exp(-(dHa - T * (dS0 + dS1)) / (mol * kB * T))
Db = Da

print("Fin")

Fin


In [3]:
# Initializes array of density values
np.random.seed(7) # Opens a random number generator instance, seed 7

rho = rho_mean * (1.0 + 0.01 * np.random.uniform(low=-1, high=1, size=(gridpoints, gridpoints))) # Creates rho values around the mean with slight randomness
rho = np.maximum(rho, 1.E-10) # Prevents negative densities

initial_mass = np.sum(rho)

@jit(nopython=True, cache=False) # Converts the given function into machine code (optimization) and tries to parallelize
def laplacian_2d(function_array):
    """
    Computes the 2D Laplacian of a function, given an array representing that function
    """
    # Note: np.roll with axis= is unsupported in Numba nopython mode
    # Periodic boundary conditions are implemented via explicit modular indexing
    n = function_array.shape[0]
    result = np.empty_like(function_array)
    for i in range(n):
        for j in range(n):
            result[i, j] = (
                function_array[i, (j+1) % n] - 2.0 * function_array[i, j] + function_array[i, (j-1) % n] +  # x-axis neighbors
                function_array[(i+1) % n, j] - 2.0 * function_array[i, j] + function_array[(i-1) % n, j]    # y-axis neighbors
            ) * inv_dx2
    return result


@jit(nopython=True, parallel=True, cache=False) # Converts the given function into machine code (optimization) and tries to parallelize
def compute_step_single(rho):
    n = rho.shape[0]

    # Total chemical potential (with floored rho, Xa)
    beta_mu_total = np.empty_like(rho)
    for i in prange(n):
        for j in range(n):
            rho_ij = rho[i, j]
            lap_rho = (
                rho[i, (j+1) % n] - 2.0 * rho_ij + rho[i, (j-1) % n] +  # x-axis neighbors
                rho[(i+1) % n, j] - 2.0 * rho_ij + rho[(i-1) % n, j]     # y-axis neighbors
            ) * inv_dx2
            beta_mu_total[i, j] = (2.0 * B2 * rho_ij + np.log(rho_ij) +                                   # beta mu_ref
                                   valence * np.log((-1 + np.sqrt(1 + 4 * 4 * rho_ij * Da)) /             # beta mu_b
                                   (2 * 4 * rho_ij * Da)) -
                                   K * lap_rho)                                                             # beta mu_int

    # Finds the 2D Laplacian of beta mu total
    laplacian_2d_mu = laplacian_2d(beta_mu_total)

    # Updates the density explicitly: rho(t+dt) = rho(t) + dt * M laplacian (beta mu_total)
    return dt * M * laplacian_2d_mu

print("Fin")

Fin


In [ ]:
# Initializes arrays for saving rho
num_saves = duration // save_interval + 1
rho_total_array = np.zeros((num_saves, gridpoints, gridpoints)) # Third dimension added for 2D grid
rho_total_array[0] = rho
save_index = 1

# Tracks the mass over time to ensure conservation
mass_history = []
time_history = []


for step in range(duration):

    # Iterates to find new value of rho
    rho += compute_step_single(rho)

    # Adds the new density to the array of densities + checks mass conservation every 10^6 steps
    if step % (save_interval) == 0:
        rho_total_array[save_index] = rho
        save_index += 1

        total_mass = np.sum(rho)

        mass_history.append(total_mass)
        time_history.append(step * dt)

        rho = np.maximum(rho, floor)

        print(f"Progress: {(step/(save_interval))} out of {duration/(save_interval)}")

Progress: 0.0 out of 250.0


In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

vmin, vmax = rho_total_array.min(), rho_total_array.max()
t_final = (save_index - 1) * save_interval * dt

im = ax.imshow(rho_total_array[save_index - 1], origin='lower', cmap='viridis',
               vmin=vmin, vmax=vmax,
               extent=[0, grid_length, 0, grid_length])
fig.colorbar(im, ax=ax, label="Density (nm⁻³)")
ax.set_title(f"t = {t_final:.2f} s")
ax.set_xlabel("x (nm)")
ax.set_ylabel("y (nm)")

plt.tight_layout()
plt.savefig("rho_final.png", dpi=150)
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

early = np.logspace(0, np.log10(20), 300, dtype=int)
late = np.logspace(np.log10(20), np.log10(save_index - 1), 100, dtype=int)
timesteps = np.unique(np.concatenate([early, late]))

data_movie = []
for t in timesteps:
    if t < rho_total_array.shape[0]:
        data_movie.append(rho_total_array[t])

vmin = np.min(data_movie)
vmax = np.max(data_movie)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(data_movie[0], origin='lower', cmap='viridis',
               vmin=vmin, vmax=vmax,
               extent=[0, grid_length, 0, grid_length])
fig.colorbar(im, ax=ax, label="Density (nm⁻³)")
ax.set_xlabel("x (nm)")
ax.set_ylabel("y (nm)")
title = ax.set_title(f"Frame: 0  |  t = 0.00 s")

def update(frame):
    im.set_data(data_movie[frame])
    t = timesteps[frame] * save_interval * dt
    title.set_text(f"Frame: {timesteps[frame]}  |  t = {t:.2f} s")
    return im, title

ani = FuncAnimation(fig, update, frames=len(data_movie), interval=200, blit=True)
ani.save("phase_separation.gif", writer=PillowWriter(fps=7))
plt.show()

In [ ]:
# CLAUDE IMPROVEMENT---INSPECT
@jit(nopython=True, parallel=True, cache=True)
def compute_step_single(rho, beta_mu_buf, lap_mu_buf):
    n = rho.shape[0]

    # Pass 1: compute beta_mu at every grid point
    for i in prange(n):
        for j in range(n):
            rho_ij = rho[i, j]
            lap_rho = (
                rho[i, (j+1) % n] - 2.0 * rho_ij + rho[i, (j-1) % n] +
                rho[(i+1) % n, j] - 2.0 * rho_ij + rho[(i-1) % n, j]
            ) * inv_dx2
            beta_mu_buf[i, j] = (
                2.0 * B2 * rho_ij
                + np.log(rho_ij)
                + valence * np.log((-1.0 + np.sqrt(1.0 + 16.0 * rho_ij * Da))
                                   / (8.0 * rho_ij * Da))
                - K * lap_rho
            )

    # Pass 2: compute Laplacian of beta_mu
    for i in prange(n):
        for j in range(n):
            mu_ij = beta_mu_buf[i, j]
            lap_mu_buf[i, j] = (
                beta_mu_buf[i, (j+1) % n] - 2.0 * mu_ij + beta_mu_buf[i, (j-1) % n] +
                beta_mu_buf[(i+1) % n, j] - 2.0 * mu_ij + beta_mu_buf[(i-1) % n, j]
            ) * inv_dx2

    return dt * M * lap_mu_buf

print("Fin")